# Assignment: Character-Level RNNs with PyTorch

**Companion reading:** Lesson 8 (Recurrent Neural Networks)

---

In this assignment you will build a **character-level language model** — one of the classic introductions to recurrent neural networks. The model reads text one character at a time and learns to predict the next character. After training on Shakespeare's writing, it can generate new text in a vaguely Shakespearean style.

This is the same family of task made famous by Andrej Karpathy's blog post [*The Unreasonable Effectiveness of Recurrent Neural Networks*](https://karpathy.github.io/2015/05/21/rnn-effectiveness/).

### Learning goals
By the end of this notebook you should be able to:
1. Explain why sequences need models with **memory** (hidden state).
2. Encode text as integer sequences for PyTorch.
3. Build and train a vanilla RNN with `torch.nn.RNN`.
4. Sample autoregressively from a trained model to generate new text.
5. Describe how **Backpropagation Through Time (BPTT)** connects to the training loop.

### What you will implement
| Part | Topic | Your work |
|------|-------|-----------|
| 1 | Setup & data | Run provided code |
| 2 | Dataset | **Complete `encode` (TODO 1)** + pass unit tests |
| 3 | Model | **Complete TODO 2** + pass unit tests |
| 4 | Training | **Complete TODO 3** + pass unit tests |
| 5 | Generation | Run & experiment |
| 6 | Reflection | Written answers |

---
## Part 0: Why RNNs?

Feedforward networks (MLPs, CNNs) treat each input independently. But language is **sequential**: the meaning of a character depends on what came before it.

A **Recurrent Neural Network (RNN)** processes a sequence step-by-step and maintains a **hidden state** $h_t$ that carries information forward:

$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b), \qquad y_t = W_y h_t + b_y$$

The same weight matrices are reused at every timestep — this is **weight sharing over time**. When we train, we **unroll** the network across timesteps and backpropagate through the entire chain. That algorithm is called **Backpropagation Through Time (BPTT)**.

In this assignment we use **character-level** modeling: each timestep is a single character, and the model predicts the next character. This keeps the vocabulary small and the task easy to inspect.

---
## Part 1: Setup

Run the cell below to import libraries and pick a compute device. If you have a GPU available, PyTorch will use it automatically.

In [ ]:
import urllib.request
import random
import time
import unittest

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Use a seed for reproducibility
SEED = 17
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def run_unit_tests(test_case_class, name=""):
    """Run a unittest TestCase and stop the notebook if any test fails."""
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(test_case_class)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        n = len(result.failures) + len(result.errors)
        raise AssertionError(f"{n} test(s) failed for {name}. Fix your implementation before continuing.")
    print(f"All {result.testsRun} tests passed for {name}.")

---
## Part 2: Load the Shakespeare Dataset

We use the **Tiny Shakespeare** corpus — about 1 MB of text, enough to train a small RNN in a few minutes on CPU.

Run the next cell to download and preview the data.

In [ ]:
SHAKESPEARE_URL = (
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
)

with urllib.request.urlopen(SHAKESPEARE_URL, timeout=15) as response:
    text = response.read().decode("utf-8")

print("Downloaded Tiny Shakespeare corpus.")
print(f"Corpus length: {len(text):,} characters")
print(f"First 300 characters:\n{text[:300]}")

### Build a character vocabulary

Neural networks work with numbers, not letters. We create two lookup tables:
- `char_to_idx`: maps each unique character → integer
- `idx_to_char`: maps integer → character

Run the cell below.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Vocabulary size: {vocab_size}")
print(f"Sample characters: {chars[:20]}")

---
## Part 3: Create Training Sequences

We train the RNN to predict the **next character** given a **context window** of previous characters.

For example, if `seq_length = 4` and the text is `"hello"`, one training pair might be:

| Input (4 chars) | Target (next char) |
|-----------------|--------------------|
| `h e l l`       | `o`                |

We slide this window across the entire corpus to create many `(input, target)` pairs.

### TODO 1 — Implement `encode`

Complete `encode` in the cell below. The `CharDataset` class is already provided — it calls `encode` to build sliding-window training pairs.

In [ ]:
def encode(s: str) -> list[int]:
    """Convert a string to a list of character indices."""
    # TODO
    pass


class CharDataset(Dataset):
    """Sliding-window dataset for next-character prediction."""

    def __init__(self, data: str, seq_length: int):
        self.seq_length = seq_length
        self.encoded = encode(data)

    def __len__(self):
        return len(self.encoded) - self.seq_length

    def __getitem__(self, idx: int):
        x = torch.tensor(self.encoded[idx : idx + self.seq_length], dtype=torch.long)
        y = torch.tensor(self.encoded[idx + self.seq_length], dtype=torch.long)
        return x, y


# --- Hyperparameters for the dataset ---
SEQ_LENGTH = 64       # characters per training window
BATCH_SIZE = 64

# Split: 90% train, 10% validation
split_idx = int(0.9 * len(text))
train_text = text[:split_idx]
val_text = text[split_idx:]

train_dataset = CharDataset(train_text, SEQ_LENGTH)
val_dataset = CharDataset(val_text, SEQ_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f"Training samples:   {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")

#### Unit tests — TODO 1

Run the cell below after implementing `encode`. All tests must pass before you continue to Part 4.

In [ ]:
class TestTODO1(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.sample = text[:50]
        cls.seq_len = 4

    def test_encode_returns_correct_indices(self):
        sample = text[:30]
        result = encode(sample)
        self.assertIsInstance(result, list)
        self.assertEqual(len(result), len(sample))
        self.assertEqual(result, [char_to_idx[c] for c in sample])

    def test_encode_empty_string(self):
        self.assertEqual(encode(""), [])

    def test_dataset_length(self):
        ds = CharDataset(self.sample, self.seq_len)
        self.assertEqual(len(ds), len(encode(self.sample)) - self.seq_len)

    def test_dataset_getitem_shapes_and_types(self):
        ds = CharDataset(self.sample, self.seq_len)
        x, y = ds[0]
        self.assertEqual(x.shape, torch.Size([self.seq_len]))
        self.assertEqual(y.shape, torch.Size([]))
        self.assertEqual(x.dtype, torch.long)
        self.assertEqual(y.dtype, torch.long)

    def test_dataset_sliding_window_content(self):
        ds = CharDataset(self.sample, self.seq_len)
        idx = 2
        x, y = ds[idx]
        self.assertEqual(x.tolist(), encode(self.sample[idx : idx + self.seq_len]))
        self.assertEqual(y.item(), encode(self.sample[idx + self.seq_len])[0])

    def test_dataset_last_valid_index(self):
        ds = CharDataset(self.sample, self.seq_len)
        idx = len(ds) - 1
        x, y = ds[idx]
        self.assertEqual(x.tolist(), encode(self.sample[idx : idx + self.seq_len]))
        self.assertEqual(y.item(), encode(self.sample[idx + self.seq_len])[0])


run_unit_tests(TestTODO1, "TODO 1")

---
## Part 4: Build the RNN Model

Our model has three parts:
1. **Embedding layer** — maps each character index to a dense vector.
2. **RNN layer** (`nn.RNN`) — processes the sequence and produces a hidden state at each step.
3. **Linear output layer** — maps hidden state → logits over the vocabulary.

```
Input chars  →  Embedding  →  RNN  →  Linear  →  logits (vocab_size)
   (B, T)         (B,T,E)     (B,T,H)   (B,T,V)
```

Where `B` = batch size, `T` = sequence length, `E` = embedding dim, `H` = hidden dim, `V` = vocab size.

### TODO 2 — Implement `CharRNN`

Fill in the `__init__` and `forward` methods below.

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim

        # TODO
        pass

    def forward(self, x, hidden=None):
        """
        x: (batch, seq_length) integer character indices
        hidden: optional initial hidden state (1, batch, hidden_dim)
        Returns: logits (batch, seq_length, vocab_size), new_hidden
        """
        # TODO
        pass

    def init_hidden(self, batch_size: int):
        """Return a zero hidden state for a new sequence."""
        return torch.zeros(1, batch_size, self.hidden_dim, device=device)


EMBED_DIM = 64
HIDDEN_DIM = 256

model = CharRNN(vocab_size, EMBED_DIM, HIDDEN_DIM).to(device)
print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

#### Unit tests — TODO 2

Run the cell below after implementing `CharRNN`. All tests must pass before you continue to Part 5.

In [ ]:
class TestTODO2(unittest.TestCase):
    def setUp(self):
        self.test_model = CharRNN(vocab_size, embed_dim=32, hidden_dim=64).to(device)

    def test_has_required_layers(self):
        self.assertIsInstance(self.test_model.embedding, nn.Embedding)
        self.assertIsInstance(self.test_model.rnn, nn.RNN)
        self.assertIsInstance(self.test_model.fc, nn.Linear)
        self.assertEqual(self.test_model.embedding.num_embeddings, vocab_size)
        self.assertEqual(self.test_model.fc.out_features, vocab_size)

    def test_forward_output_shapes(self):
        batch_size, seq_len = 4, 16
        x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
        logits, hidden = self.test_model(x)
        self.assertEqual(logits.shape, (batch_size, seq_len, vocab_size))
        self.assertEqual(hidden.shape, (1, batch_size, 64))

    def test_forward_with_provided_hidden_state(self):
        batch_size, seq_len = 2, 8
        x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
        h0 = self.test_model.init_hidden(batch_size)
        logits, h1 = self.test_model(x, h0)
        self.assertEqual(logits.shape, (batch_size, seq_len, vocab_size))
        self.assertEqual(h1.shape, h0.shape)

    def test_init_hidden_shape(self):
        h = self.test_model.init_hidden(3)
        self.assertEqual(h.shape, (1, 3, 64))


run_unit_tests(TestTODO2, "TODO 2")

---
## Part 5: Train the Model

Training loop overview:
1. Forward pass: get logits for every character in the batch.
2. Compute **cross-entropy loss** between predicted logits and true next characters.
3. Backward pass (BPTT): `loss.backward()` propagates gradients through the unrolled RNN.
4. Optimizer step: update weights.

We also track **perplexity** — a standard language-modeling metric. Lower is better. Perplexity ≈ `exp(cross_entropy_loss)`.

### TODO 3 — Implement `train_one_epoch` and `evaluate`

Complete the training and evaluation functions below.

import math

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        # TODO

        batch_tokens = x.numel()
        total_loss += loss.item() * batch_tokens
        total_tokens += batch_tokens

    return total_loss / total_tokens


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # TODO

        batch_tokens = x.numel()
        total_loss += loss.item() * batch_tokens
        total_tokens += batch_tokens

    return total_loss / total_tokens


def loss_to_ppl(loss):
    return math.exp(loss)

#### Unit tests — TODO 3

Run the cell below after implementing `train_one_epoch` and `evaluate`. All tests must pass before you run full training.

In [ ]:
class TestTODO3(unittest.TestCase):
    def setUp(self):
        self.test_model = CharRNN(vocab_size, EMBED_DIM, HIDDEN_DIM).to(device)
        self.test_optimizer = torch.optim.Adam(self.test_model.parameters(), lr=0.003)
        self.tiny_loader = DataLoader(
            CharDataset(text[:400], seq_length=8),
            batch_size=4,
            shuffle=False,
            drop_last=True,
        )

    def test_train_one_epoch_returns_finite_loss(self):
        loss = train_one_epoch(self.test_model, self.tiny_loader, self.test_optimizer, criterion)
        self.assertIsInstance(loss, float)
        self.assertFalse(math.isnan(loss))
        self.assertFalse(math.isinf(loss))
        self.assertGreater(loss, 0.0)

    def test_evaluate_returns_finite_loss(self):
        loss = evaluate(self.test_model, self.tiny_loader, criterion)
        self.assertIsInstance(loss, float)
        self.assertFalse(math.isnan(loss))
        self.assertFalse(math.isinf(loss))
        self.assertGreater(loss, 0.0)

    def test_training_updates_weights(self):
        before = [p.detach().clone() for p in self.test_model.parameters()]
        train_one_epoch(self.test_model, self.tiny_loader, self.test_optimizer, criterion)
        after = [p.detach() for p in self.test_model.parameters()]
        changed = any(not torch.allclose(b, a) for b, a in zip(before, after))
        self.assertTrue(changed, "Expected at least one model parameter to change after training")

    def test_evaluate_does_not_change_weights(self):
        before = [p.detach().clone() for p in self.test_model.parameters()]
        evaluate(self.test_model, self.tiny_loader, criterion)
        after = [p.detach() for p in self.test_model.parameters()]
        for b, a in zip(before, after):
            self.assertTrue(torch.allclose(b, a))


run_unit_tests(TestTODO3, "TODO 3")

Run the training cell below. On CPU this takes roughly 2–5 minutes for 5 epochs.

In [ ]:
NUM_EPOCHS = 5

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss = evaluate(model, val_loader, criterion)
    elapsed = time.time() - t0

    print(
        f"Epoch {epoch}/{NUM_EPOCHS} | "
        f"train loss: {train_loss:.4f} (ppl {loss_to_ppl(train_loss):.1f}) | "
        f"val loss: {val_loss:.4f} (ppl {loss_to_ppl(val_loss):.1f}) | "
        f"{elapsed:.1f}s"
    )

---
## Part 6: Generate Shakespeare-Like Text

Now we **sample** from the model autoregressively:
1. Start with a seed string (e.g. `"ROMEO:"`).
2. Feed the last `SEQ_LENGTH` characters to the model.
3. Sample the next character from the output probability distribution.
4. Append it to the seed and repeat.

Run the cell below to try out your model! Experiment with its output by changing `seed` and `length`.

In [ ]:
@torch.no_grad()
def generate(model, seed: str, length: int = 400, temperature: float = 0.8) -> str:
    """
    Autoregressively generate `length` new characters.

    temperature: lower = more conservative, higher = more random
    """
    model.eval()
    generated = seed
    hidden = model.init_hidden(1)

    # Prime the hidden state with the seed (process it in chunks)
    seed_encoded = encode(seed)
    if len(seed_encoded) >= SEQ_LENGTH:
        prime = torch.tensor([seed_encoded[-SEQ_LENGTH:]], dtype=torch.long, device=device)
    else:
        prime = torch.tensor([seed_encoded], dtype=torch.long, device=device)

    logits, hidden = model(prime, hidden)

    last_char_idx = seed_encoded[-1]

    for _ in range(length):
        x = torch.tensor([[last_char_idx]], dtype=torch.long, device=device)
        logits, hidden = model(x, hidden)
        logits = logits[:, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1).item()
        generated += idx_to_char[next_idx]
        last_char_idx = next_idx

    return generated


sample = generate(model, seed="ROMEO:\n", length=500, temperature=0.8)
print(sample)

### Experiment: temperature

Try generating with different temperatures and compare the output:

```python
for temp in [0.5, 0.8, 1.2]:
    print(f"\n--- temperature={temp} ---")
    print(generate(model, seed="KING:\n", length=200, temperature=temp))
```

**Question:** What happens to the text as temperature increases? Answer in the markdown below.

---
## Part 8: Reflection Questions

Answer these in the markdown cell below.

1. **Hidden state:** In your own words, what role does the hidden state play during both training and generation?

2. **Weight sharing:** Why does an RNN use the same weights at every timestep? What would happen if each timestep had its own separate weights?

3. **BPTT:** When `loss.backward()` is called, gradients flow backward through all timesteps in the batch. Why can gradients become very small for early timesteps in a vanilla RNN?

4. **Character vs. word level:** What are the trade-offs of modeling text one character at a time versus one word at a time?

5. **Generated output:** Paste your favorite generated sample. What patterns did the model learn? What did it fail to learn?